## Data preprocessing

In [2]:
## Defining paths
import os
from pathlib import Path

# GitHub details
URL_REPO = "https://github.com/DataScientest-Studio/oct25_bmlops_int_rakuten.git"
BRANCH = "phase_1"

# path on GoogleDrive Colab
# ROOT = Path("/content/drive/MyDrive/1_Projekte_Datensätze/1_Projekte/2_Rakuten_Classification")               
DATA = Path("/mnt/chromeos/GoogleDrive/MyDrive/1_Projekte_Datensätze/1_Projekte/2_Rakuten_Classification/data")    
DATA.mkdir(parents=True, exist_ok=True)

# local path on my Chromebook
ROOT = Path("/home/robfra/0_Portfolio_Projekte/Rakuten_Classification")

# further paths  
DATA_PROCESSED = Path(DATA, "processed")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

DATA_RAW = Path(DATA, "raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

# VENV_PATH = "/home/robfra/_venv/.venv_rakuten"


In [ ]:
## load data (from MongoDB)
import pandas as pd 

df_names = ["df_test", "df_train", 
           # "Y_train_CVw08PX"
           ]

to_embed = {}
# required_cols = ["clean_designation", "clean_description", "productid"]

for name in df_names:
    print(f"START processing {name}...")
    df = pd.read_csv(f"{DATA_PROCESSED}/{name}.csv") # , index_col=0)
    
    # available = [col for col in required_cols if col in df.columns]
    # if not available:
    #     raise ValueError(f"None of the required columns {required_cols} are present in {f}_clean.csv")
    
    # for col in required_cols:
    #     if col not in df.columns:
    #         df[col] = ""  # or handle missing columns as needed

    # clean_df = df[required_cols].copy() #.fillna("") # if "clean_designation" in df.columns
    for col in ["clean_designation", "clean_description"]:
        df[col].fillna("")

        # clean_desc = df[[]].copy if "clean_description" in df.columns
        df["text"] = (
            df["clean_designation"].astype(str).str.strip()
            + " "
            + df["clean_description"].astype(str).str.strip()
                    ).str.strip()
        
    to_embed[name] = df
    print(f"FINISHED processing {name}.\n")


In [ ]:

# OpenAI Embeddings, fastText, spaCy etc.
from sentence_transformers import SentenceTransformer
import numpy as np
# from tqdm import tqdm
from rich.progress import Progress

# Wenn du bestmögliche Embedding-Qualität willst → intfloat/E5 oder BGE (BAAI/bge-small-en oder bge-base-en)

# „Best-of-all“ für Produktdaten:
# intfloat/e5-base
# oder
# e5-large (zu langsam auf CPU)

# Wenn deine Produkttexte DE/FR/… enthalten → Modell performt schlechter.
# Dann brauchst du:
# distiluse-base-multilingual-cased-v2
# oder sentence-transformers/paraphrase-multilingual-mpnet-base-v2
# oder intfloat/multilingual-e5-small / base

for name, df in to_embed.items():
    # df.reset_index(drop=True, inplace=True)
    model = SentenceTransformer('all-MiniLM-L6-v2')
    texts = df["text"].tolist()

    embeddings = []
    batch_size = 256

    with Progress() as progress:
        task = progress.add_task(f"Start embedding for {name} with {len(texts)} texts...", total=len(texts))
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            emb = model.encode(batch, convert_to_numpy=True, normalize_embeddings=True)
            embeddings.append(emb)
            progress.update(task, advance=len(batch))

        # print(f"Start embedding for {name} with {len(texts)} texts...")
        # for i in tqdm(range(0, len(texts), batch_size)):
        #     batch = texts[i:i+batch_size]
        #     emb = model.encode(batch, convert_to_numpy=True, normalize_embeddings=True)
        #     embeddings.append(emb)

    embeddings = np.vstack(embeddings)

    print(f"--> embeddings shape:\t", embeddings.shape)

    df["embed_text"] = list(embeddings)

    try:
        df.to_csv(f"{DATA_PROCESSED}/{name}_embedded.csv", index=False)
        print(f"Saved embeddings to {DATA_PROCESSED}/{name}_embedded.csv")
    except Exception as e:
        print(f"Error saving embeddings for {name}: {e}")
# embed_designation = model.encode(dict_dfs["X_train_update"]["clean_designation"].tolist())

# Start embedding for X_test_update with 13812 texts...
# 100%|██████████| 54/54 [30:25<00:00, 33.80s/it]
# --> embeddings shape:	 (13812, 384)
# Saved embeddings to /mnt/chromeos/GoogleDrive/MyDrive/1_Projekte_Datensätze/1_Projekte/2_Rakuten_Classification/data/processed/X_test_update_embedded.csv
# Start embedding for X_train_update with 84916 texts...
# 100%|██████████| 332/332 [3:07:24<00:00, 33.87s/it]  
# --> embeddings shape:	 (84916, 384)

Output()

: 

In [ ]:
/home/robfra/0_Portfolio_Projekte/Rakuten_Classification/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

/home/robfra/0_Portfolio_Projekte/Rakuten_Classification/.venv/lib/python3.12/site-packages/rich/live.py:256: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')